In [ ]:
from scipy.interpolate import griddata
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import csv
from matplotlib.lines import Line2D
from scipy.integrate import quad
from functools import reduce
import operator
Cww = 1
Cgg = 0
Cbb = -1
mz = 91.188
mw = 80.3692
cw = mw/mz
sw = np.sqrt(1 - cw**2) 
alpha = 1/128
alpha_s = 0.11 #mz
hbar_c = 1.973e-14  # GeV·cm

Caa = Cww + Cbb
Caz = cw**2 * Cww - sw**2 * Cbb
Czz = cw**4 * Cww + sw**4 * Cbb

In [ ]:
L1 = 25 #3025
L2 = 150 #ab-1 #7407

In [ ]:
ma = [0.001, 0.01, 0.1, 1, 10]
S_eff = [0.991, 0.991, 0.991, 0.991, 0.989]
B_eff = [0.026, 0.026, 0.026, 0.026, 0.026]
B_cs = 14.01627324

# 读取数据
pho_ALP = pd.read_csv('./alp_decay_widths_had2.csv')

# 将数据类型转换为更小的浮动数类型以减少内存使用
pho_ALP['xs_ME'] =pho_ALP['xs_ME'].astype('float32')
pho_ALP['Lambda'] = 1000 / pho_ALP['Lambda'].astype('float32')

xs = pho_ALP['xs_ME'].astype('float32')
gstar = pho_ALP['Lambda'].astype('float32')
MQ = pho_ALP['ma'].astype('float32')
# 为信号截面插值，生成从7到79GeV每隔2GeV的数据点
signal_xs_interp = np.interp(pho_ALP['ma'].astype('float32'), ma, S_eff)
# 创建网格，减少网格密度以降低内存使用
mq_14_os = np.logspace(np.log10(pho_ALP['ma'].min()), np.log10(pho_ALP['ma'].max()), 2000)
gstar_14_os = np.logspace(np.log10(gstar.min()), np.log10(gstar.max()), 200)

# 计算246/gstar的y轴
y_axis = gstar_14_os

# 将y_axis转换为与MQ_grid_14_os相同的形状
y_axis_grid = np.tile(y_axis, (mq_14_os.shape[0], 1))  # 广播到2D

# 创建网格
MQ_grid_14_osme, Gstar_grid_14_osme = np.meshgrid(mq_14_os, gstar_14_os)

# 使用二维插值来计算网格上的 S 和 Z
points_os = MQ,gstar.values
values_os = xs.values


# 将效率的插值结果重塑为(100, 100)形状
signal_efficiency_grid = 0.999
background_efficiency_grid = 0.00038

# 重新计算S_grid和背景
S_grid_14_os_ME = griddata(points_os, values_os, (MQ_grid_14_osme, Gstar_grid_14_osme), method='linear') * L1*1000 * 0.999 * 1000  
background_14_os_ME = griddata(points_os, tuple([B_cs]*np.shape(values_os)[0]), (MQ_grid_14_osme, Gstar_grid_14_osme), method='linear') * L1*1000 * 0.00038 * 1000  

# 计算显著度Z = S / sqrt(S + B)
Z_grid_14_os_ME = S_grid_14_os_ME / np.sqrt(S_grid_14_os_ME + background_14_os_ME)

# 创建30fb情况
# 用30fb进行计算
S_grid_14_os_30_ME = griddata(points_os, values_os, (MQ_grid_14_osme, Gstar_grid_14_osme), method='linear') * L2*1000 * signal_efficiency_grid * 1000  
background_14_os_30_ME = griddata(points_os, tuple([B_cs]*np.shape(values_os)[0]), (MQ_grid_14_osme, Gstar_grid_14_osme), method='linear') * L2*1000 * background_efficiency_grid * 1000 


# 计算显著度Z = S / sqrt(S + B)
Z_grid_14_os_30_ME = S_grid_14_os_30_ME / np.sqrt(S_grid_14_os_30_ME + background_14_os_30_ME)

def sigma_Z(s, b):
    term1 = (s**2 + 4*b*s + 4*b**2)*s / (4 * (s + b)**3)
    term2 = (s**2 * b) / (4 * (s + b)**3)
    return np.sqrt(term1 + term2)

std_3_ME=sigma_Z(S_grid_14_os_ME,background_14_os_ME)
z3MEp = Z_grid_14_os_ME+np.sqrt(std_3_ME)
z3MEm = Z_grid_14_os_ME-np.sqrt(std_3_ME)

std_30_ME=sigma_Z(S_grid_14_os_30_ME,background_14_os_30_ME)
z30MEp = Z_grid_14_os_30_ME+np.sqrt(std_30_ME)
z30MEm = Z_grid_14_os_30_ME-np.sqrt(std_30_ME)
# 绘制单张图包含两个等值线图
plt.figure(figsize=(8, 6))


contour1 = plt.contour(MQ_grid_14_osme, Gstar_grid_14_osme, Z_grid_14_os_30_ME, levels=[2], colors='darkorange', label='Z_grid_14_os_30')
#plt.clabel(contour1, fmt={2: fmt2}, inline=True, fontsize=10)
contour_ME30p = plt.contour(MQ_grid_14_osme, Gstar_grid_14_osme, z30MEp, levels=[2], colors='darkorange', label='Z_grid_14_os_30p',linestyles='--')
contour_ME30m = plt.contour(MQ_grid_14_osme, Gstar_grid_14_osme, z30MEm, levels=[2], colors='darkorange', label='Z_grid_14_os_30m',linestyles='--')
upper_segments_30 = contour_ME30p.allsegs[0]
lower_segments_30 = contour_ME30m.allsegs[0]
for upper_segment, lower_segment in zip(upper_segments_30, lower_segments_30):
    upper_coords = np.array(upper_segment)
    lower_coords = np.array(lower_segment)
    full_coords = np.vstack([upper_coords, lower_coords[::-1]])
    plt.fill(full_coords[:, 0], full_coords[:, 1], color='#ffc680', alpha=0.5)

# Z_grid_14_os
contour2 = plt.contour(MQ_grid_14_osme, Gstar_grid_14_osme, Z_grid_14_os_ME, levels=[2], colors='navy', label='Z_grid_14_os')
#plt.clabel(contour2, fmt={2: fmt1}, inline=True, fontsize=10,manual=[(70,2e-3)])
contour_ME3p = plt.contour(MQ_grid_14_osme, Gstar_grid_14_osme, z3MEp, levels=[2], colors='navy', label='Z_grid_14_os_30p',linestyles='--')
contour_ME3m = plt.contour(MQ_grid_14_osme, Gstar_grid_14_osme, z3MEm, levels=[2], colors='navy', label='Z_grid_14_os_30m',linestyles='--')
upper_segments_3 = contour_ME3p.allsegs[0]
lower_segments_3 = contour_ME3m.allsegs[0]
for upper_segment, lower_segment in zip(upper_segments_3, lower_segments_3):
    upper_coords = np.array(upper_segment)
    lower_coords = np.array(lower_segment)
    full_coords = np.vstack([upper_coords, lower_coords[::-1]])
    plt.fill(full_coords[:, 0], full_coords[:, 1], color='#8080c0', alpha=0.5)

#plt.title(r'ALP$\rightarrow \tau\tau \rightarrow e \mu $')
plt.ylim(1e-4,1e0)

plt.xlabel(R"m$_{a}$ [GeV]", fontsize=20)
plt.ylabel(R"$1/\Lambda$ [TeV]$^{-1}$", fontsize=20)
plt.yscale('log')
plt.xscale('log')
plt.tick_params(axis='x', labelsize=16)  # x 轴刻度字体大小
plt.tick_params(axis='y', labelsize=16)  # y 轴刻度字体大小
plt.grid()
legend_elements = [
    Line2D([0], [0], color='navy', lw=2, label=R'2$\sigma$ C.L. 25 ab$^{-1}$'),
    Line2D([0], [0], color='darkorange', lw=2, label=R'2$\sigma$ C.L. 150 ab$^{-1}$ '),
    Line2D([0], [0], color='navy', lw=2, linestyle='--', label=R'25 ab$^{-1}$ 1$\sigma$ Error boundary'),
    Line2D([0], [0], color='darkorange', lw=2, linestyle='--', label=R'150 ab$^{-1}$ 1$\sigma$ Error boundary')
]
plt.text(0.45, 0.8, R'Mono-$\gamma$', ha='center', va='center', transform=plt.gca().transAxes, fontsize=18, color='black')
plt.legend(handles=legend_elements, loc='best', fontsize=10)
#plt.savefig('./aME.png',dpi=300)
plt.show()

In [ ]:
ma = [10, 30, 50, 70, 85]
S_eff = [0.62, 0.768, 0.732, 0.581, 0.162]
B_eff = [0.035, 0.035, 0.035, 0.035, 0.035]
B_cs = 84.29*0.178*0.174
mtau2=1.777*2
# 读取数据
pho_ALP = pd.read_csv('./alp_decay_widths_had2.csv')
# 将数据类型转换为更小的浮动数类型以减少内存使用
pho_ALP_tautau = 0.178*0.174*pho_ALP['xs_tautau_prompt'].astype('float32')
pho_ALP_lambda = 1000 / pho_ALP['Lambda']

xs = pho_ALP_tautau
gstar = pho_ALP_lambda
MQ = pho_ALP['ma']
# 为信号截面插值，生成从7到79GeV每隔2GeV的数据点
signal_xs_interp = np.interp(pho_ALP['ma'], ma, S_eff)

In [ ]:
# 创建网格，减少网格密度以降低内存使用
mq_14_os = np.logspace(np.log10(pho_ALP['ma'].min()), np.log10(pho_ALP['ma'].max()), 2000)
gstar_14_os = np.logspace(np.log10(gstar.min()), np.log10(gstar.max()), 200)
#mq_14_os = np.linspace(pho_ALP['ma'].min(), pho_ALP['ma'].max(), 2000)
#gstar_14_os = np.linspace(gstar.min(), gstar.max(), 200)
# 创建网格
MQ_grid_14_os, Gstar_grid_14_os = np.meshgrid(mq_14_os, gstar_14_os)

# 构造插值用的点和目标点
points_frac = np.vstack((MQ, gstar)).T

# 使用二维插值来计算网格上的 S 和 Z
points_os = MQ,gstar.values
values_os = xs.values

# 对每个点进行信号和背景效率的插值
signal_efficiency = np.interp(MQ_grid_14_os.flatten(), ma, S_eff)
background_efficiency = np.interp(MQ_grid_14_os.flatten(), ma, B_eff)

# 将效率的插值结果重塑为(100, 100)形状
signal_efficiency_grid = signal_efficiency.reshape(MQ_grid_14_os.shape)
background_efficiency_grid = background_efficiency.reshape(MQ_grid_14_os.shape)


# 重新计算S_grid和背景
S_grid_14_os_emu = griddata(points_os, values_os, (MQ_grid_14_os, Gstar_grid_14_os), method='linear')* L1*1000 * signal_efficiency_grid * 1000  # 25fb
background_14_os_emu = griddata(points_os, tuple([B_cs]*np.shape(values_os)[0]), (MQ_grid_14_os, Gstar_grid_14_os), method='linear') * L1*1000 * background_efficiency_grid * 1000  # 25fb

# 计算显著度Z = S / sqrt(S + B)
Z_grid_14_os_emu = S_grid_14_os_emu / np.sqrt(S_grid_14_os_emu + background_14_os_emu)

# 创建30fb情况
# 用30fb进行计算
S_grid_14_os_30_emu = griddata(points_os, values_os, (MQ_grid_14_os, Gstar_grid_14_os), method='linear') * L2*1000 * signal_efficiency_grid * 1000  
background_14_os_30_emu = griddata(points_os, tuple([B_cs]*np.shape(values_os)[0]), (MQ_grid_14_os, Gstar_grid_14_os), method='linear') * L2*1000 * background_efficiency_grid * 1000 


# 计算显著度Z = S / sqrt(S + B)
Z_grid_14_os_30_emu = S_grid_14_os_30_emu / np.sqrt(S_grid_14_os_30_emu + background_14_os_30_emu)

def sigma_Z(s, b):
    term1 = (s**2 + 4*b*s + 4*b**2)*s / (4 * (s + b)**3)
    term2 = (s**2 * b) / (4 * (s + b)**3)
    return np.sqrt(term1 + term2)

std_3_emu=sigma_Z(S_grid_14_os_emu,background_14_os_emu)
#Z_3_amu= Z_grid_14_os_amu / np.sqrt(std_3_amu)
z3emup = Z_grid_14_os_emu+np.sqrt(std_3_emu)
z3emum = Z_grid_14_os_emu-np.sqrt(std_3_emu)

std_30_emu=sigma_Z(S_grid_14_os_30_emu,background_14_os_30_emu)
#Z_30_amu= Z_grid_14_os_30_amu / np.sqrt(std_30_amu)
z30emup = Z_grid_14_os_30_emu+np.sqrt(std_30_emu)
z30emum = Z_grid_14_os_30_emu-np.sqrt(std_30_emu)

# 绘制单张图包含两个等值线图
plt.figure(figsize=(8, 6))

# Z_grid_14_os_30
contour1 = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, Z_grid_14_os_30_emu, levels=[2], colors='darkorange')
#plt.clabel(contour1, fmt={2: fmt2}, inline=True, fontsize=10)
contour_emu30p = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, z30emup, levels=[2], colors='darkorange',linestyles='--')
contour_emu30m = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, z30emum, levels=[2], colors='darkorange',linestyles='--')
upper_segments_30 = contour_emu30p.allsegs[0]
lower_segments_30 = contour_emu30m.allsegs[0]
for upper_segment, lower_segment in zip(upper_segments_30, lower_segments_30):
    upper_coords = np.array(upper_segment)
    lower_coords = np.array(lower_segment)
    full_coords = np.vstack([upper_coords, lower_coords[::-1]])
    plt.fill(full_coords[:, 0], full_coords[:, 1], color='#ffc680', alpha=0.5)

# Z_grid_14_os
contour2 = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, Z_grid_14_os_emu, levels=[2], colors='navy')
#plt.clabel(contour2, fmt={2: fmt1}, inline=True, fontsize=10,manual=[(70,2e-3)])
contour_emu3p = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, z3emup, levels=[2], colors='navy',linestyles='--')
contour_emu3m = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, z3emum, levels=[2], colors='navy',linestyles='--')
upper_segments_3 = contour_emu3p.allsegs[0]
lower_segments_3 = contour_emu3m.allsegs[0]
for upper_segment, lower_segment in zip(upper_segments_3, lower_segments_3):
    upper_coords = np.array(upper_segment)
    lower_coords = np.array(lower_segment)
    full_coords = np.vstack([upper_coords, lower_coords[::-1]])
    plt.fill(full_coords[:, 0], full_coords[:, 1], color='#8080c0', alpha=0.5)

#plt.title(r'ALP$\rightarrow \tau\tau \rightarrow e \mu $')
plt.ylim(1e-2,1e1)
plt.xlim(1,92)
plt.xlabel(R"m$_{a}$ [GeV]", fontsize=20)
plt.ylabel(R"$1/\Lambda$ [TeV]$^{-1}$", fontsize=20)
plt.axvline(x=mtau2, linewidth=1.5)
plt.yscale('log')
plt.xscale('log')
plt.tick_params(axis='x', labelsize=16)  # x 轴刻度字体大小
plt.tick_params(axis='y', labelsize=16)  # y 轴刻度字体大小
plt.grid()
legend_elements = [
    Line2D([0], [0], color='navy', lw=2, label=R'2$\sigma$ C.L. 25 ab$^{-1}$'),
    Line2D([0], [0], color='darkorange', lw=2, label=R'2$\sigma$ C.L. 150 ab$^{-1}$ '),
    Line2D([0], [0], color='navy', lw=2, linestyle='--', label=R'25 ab$^{-1}$ 1$\sigma$ Error boundary'),
    Line2D([0], [0], color='darkorange', lw=2, linestyle='--', label=R'150 ab$^{-1}$ 1$\sigma$ Error boundary')
]
plt.text(0.6, 0.8, R'$a \to \tau^{+}(\tau^{+}\to \mu^{+} \bar{v}_{\tau} v_{\mu}) \tau^{-}(\tau^{-}\to e^{-} v_{\tau} \bar{v}_{e})$', ha='center', va='center', transform=plt.gca().transAxes, fontsize=14, color='black')
plt.legend(handles=legend_elements, loc='best', fontsize=10)
plt.savefig('./fignew/tauemu.png',dpi=300)
plt.show()

In [ ]:
ma = [10, 30, 50, 70, 85]
S_eff = [0.921, 0.807, 0.756, 0.599, 0.234]
B_eff = [0.04, 0.04, 0.04, 0.04, 0.04]
B_cs = 84.29*0.174**2

# 读取数据
pho_ALP = pd.read_csv('./alp_decay_widths_had2.csv')

# 将数据类型转换为更小的浮动数类型以减少内存使用
pho_ALP_tautau = 0.174*0.174*pho_ALP['xs_tautau_prompt'].astype('float32')
pho_ALP_lambda = 1000 / pho_ALP['Lambda']

xs = pho_ALP_tautau
gstar = pho_ALP_lambda
MQ = pho_ALP['ma']
# 为信号截面插值，生成从7到79GeV每隔2GeV的数据点
signal_xs_interp = np.interp(pho_ALP['ma'], ma, S_eff)

# 创建网格，减少网格密度以降低内存使用
mq_14_os = np.logspace(np.log10(pho_ALP['ma'].min()), np.log10(pho_ALP['ma'].max()), 2000)
gstar_14_os = np.logspace(np.log10(gstar.min()), np.log10(gstar.max()), 200)

# 创建网格
MQ_grid_14_os, Gstar_grid_14_os = np.meshgrid(mq_14_os, gstar_14_os)


# 使用二维插值来计算网格上的 S 和 Z
points_os = MQ,gstar.values
values_os = xs.values

# 对每个点进行信号和背景效率的插值
signal_efficiency = np.interp(MQ_grid_14_os.flatten(), ma, S_eff)
background_efficiency = np.interp(MQ_grid_14_os.flatten(), ma, B_eff)

# 将效率的插值结果重塑为(100, 100)形状
signal_efficiency_grid = signal_efficiency.reshape(MQ_grid_14_os.shape)
background_efficiency_grid = background_efficiency.reshape(MQ_grid_14_os.shape)


# 重新计算S_grid和背景
S_grid_14_os_e = griddata(points_os, values_os, (MQ_grid_14_os, Gstar_grid_14_os), method='linear') * L1*1000 * signal_efficiency_grid * 1000  # 25fb
background_14_os_e = griddata(points_os, tuple([B_cs]*np.shape(values_os)[0]), (MQ_grid_14_os, Gstar_grid_14_os), method='linear') * L1*1000 * background_efficiency_grid * 1000  # 25fb

# 计算显著度Z = S / sqrt(S + B)
Z_grid_14_os_e = S_grid_14_os_e / np.sqrt(S_grid_14_os_e + background_14_os_e)

# 创建30fb情况
# 用30fb进行计算
S_grid_14_os_30_e = griddata(points_os, values_os, (MQ_grid_14_os, Gstar_grid_14_os), method='linear') * L2*1000 * signal_efficiency_grid * 1000  
background_14_os_30_e = griddata(points_os, tuple([B_cs]*np.shape(values_os)[0]), (MQ_grid_14_os, Gstar_grid_14_os), method='linear') * L2*1000 * background_efficiency_grid * 1000 


# 计算显著度Z = S / sqrt(S + B)
Z_grid_14_os_30_e = S_grid_14_os_30_e / np.sqrt(S_grid_14_os_30_e + background_14_os_30_e)

def sigma_Z(s, b):
    term1 = (s**2 + 4*b*s + 4*b**2)*s / (4 * (s + b)**3)
    term2 = (s**2 * b) / (4 * (s + b)**3)
    return np.sqrt(term1 + term2)

std_3_e=sigma_Z(S_grid_14_os_e,background_14_os_e)
#Z_3_amu= Z_grid_14_os_amu / np.sqrt(std_3_amu)
z3ep = Z_grid_14_os_e+np.sqrt(std_3_e)
z3em = Z_grid_14_os_e-np.sqrt(std_3_e)

std_30_e=sigma_Z(S_grid_14_os_30_e,background_14_os_30_e)
#Z_30_amu= Z_grid_14_os_30_amu / np.sqrt(std_30_amu)
z30ep = Z_grid_14_os_30_e+np.sqrt(std_30_e)
z30em = Z_grid_14_os_30_e-np.sqrt(std_30_e)

# 绘制单张图包含两个等值线图
plt.figure(figsize=(8, 6))

# Signal yield
N_prompt_150 = xs  * 150 * 10**(6)  # pb \to ab
N_prompt_25 = xs * 25 * 10**(6) # pb \to ab
# Plot settings
labels = [R'Prompt: 150 ab$^{-1}$', R'Prompt: 25 ab$^{-1}$']
handles = []

plt.figure(figsize=(8,6))
N_prompt_150_grid = griddata(points_os, N_prompt_150.values, (MQ_grid_14_os, Gstar_grid_14_os), method='linear')
N_prompt_25_grid = griddata(points_os, N_prompt_25.values, (MQ_grid_14_os, Gstar_grid_14_os), method='linear')



# Z_grid_14_os_30
contour1 = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, Z_grid_14_os_30_e, levels=[2], colors='darkorange')
#plt.clabel(contour1, fmt={2: fmt2}, inline=True, fontsize=10)
contour_e30p = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, z30ep, levels=[2], colors='darkorange',linestyles='--')
contour_e30m = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, z30em, levels=[2], colors='darkorange',linestyles='--')
upper_segments_30 = contour_e30p.allsegs[0]
lower_segments_30 = contour_e30m.allsegs[0]
for upper_segment, lower_segment in zip(upper_segments_30, lower_segments_30):
    upper_coords = np.array(upper_segment)
    lower_coords = np.array(lower_segment)
    full_coords = np.vstack([upper_coords, lower_coords[::-1]])
    plt.fill(full_coords[:, 0], full_coords[:, 1], color='#ffc680', alpha=0.5)

# Z_grid_14_os
contour2 = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, Z_grid_14_os_e, levels=[2], colors='navy')
#plt.clabel(contour2, fmt={2: fmt1}, inline=True, fontsize=10,manual=[(70,2e-3)])
contour_e3p = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, z3ep, levels=[2], colors='navy',linestyles='--')
contour_e3m = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, z3em, levels=[2], colors='navy',linestyles='--')
upper_segments_3 = contour_e3p.allsegs[0]
lower_segments_3 = contour_e3m.allsegs[0]
for upper_segment, lower_segment in zip(upper_segments_3, lower_segments_3):
    upper_coords = np.array(upper_segment)
    lower_coords = np.array(lower_segment)
    full_coords = np.vstack([upper_coords, lower_coords[::-1]])
    plt.fill(full_coords[:, 0], full_coords[:, 1], color='#8080c0', alpha=0.5)

#plt.title(r'ALP$\rightarrow \tau\tau \rightarrow e \mu $')
plt.ylim(1e-2,1e1)
plt.xlim(1e0,92)
plt.xlabel(R"m$_{a}$ [GeV]", fontsize=20)
plt.ylabel(R"$1/\Lambda$ [TeV]$^{-1}$", fontsize=20)
plt.axvline(x=mtau2, linewidth=1.5)
plt.yscale('log')
plt.xscale('log')
plt.tick_params(axis='x', labelsize=16)  # x 轴刻度字体大小
plt.tick_params(axis='y', labelsize=16)  # y 轴刻度字体大小
plt.grid()
legend_elements = [
    Line2D([0], [0], color='navy', lw=2, label=R'2$\sigma$ C.L. 25 ab$^{-1}$'),
    Line2D([0], [0], color='darkorange', lw=2, label=R'2$\sigma$ C.L. 150 ab$^{-1}$ '),
    Line2D([0], [0], color='navy', lw=2, linestyle='--', label=R'25 ab$^{-1}$ 1$\sigma$ Error boundary'),
    Line2D([0], [0], color='darkorange', lw=2, linestyle='--', label=R'150 ab$^{-1}$ 1$\sigma$ Error boundary')
]
plt.text(0.6, 0.7, R'$a \to \tau^{+}(\tau^{+}\to e^{+} \bar{v}_{\tau} v_{e}) \tau^{-}(\tau^{-}\to e^{-} v_{\tau} \bar{v}_{e})$', ha='center', va='center', transform=plt.gca().transAxes, fontsize=14, color='black')
plt.legend(handles=legend_elements, loc='best', fontsize=10)
plt.savefig('./fignew/tauee.png',dpi=300)
plt.show()

In [ ]:
ma = [10, 30, 50, 70, 85]
S_eff = [0.923, 0.797, 0.755, 0.623, 0.239]
B_eff = [0.04, 0.04, 0.04, 0.04, 0.04]
B_cs = 84.29*0.178**2

# 读取数据
pho_ALP = pd.read_csv('./alp_decay_widths_had2.csv')

# 将数据类型转换为更小的浮动数类型以减少内存使用
pho_ALP_tautau = 0.178*0.178*pho_ALP['xs_tautau_prompt'].astype('float32')
pho_ALP_lambda = 1000 / pho_ALP['Lambda']

xs = pho_ALP_tautau
gstar = pho_ALP_lambda
MQ = pho_ALP['ma']
# 为信号截面插值，生成从7到79GeV每隔2GeV的数据点
signal_xs_interp = np.interp(pho_ALP['ma'], ma, S_eff)

# 创建网格，减少网格密度以降低内存使用
mq_14_os = np.logspace(np.log10(pho_ALP['ma'].min()), np.log10(pho_ALP['ma'].max()), 2000)
gstar_14_os = np.logspace(np.log10(gstar.min()), np.log10(gstar.max()), 200)

# 创建网格
MQ_grid_14_os, Gstar_grid_14_os = np.meshgrid(mq_14_os, gstar_14_os)

# 使用二维插值来计算网格上的 S 和 Z
points_os = MQ,gstar.values
values_os = xs.values

# 对每个点进行信号和背景效率的插值
signal_efficiency = np.interp(MQ_grid_14_os.flatten(), ma, S_eff)
background_efficiency = np.interp(MQ_grid_14_os.flatten(), ma, B_eff)

# 将效率的插值结果重塑为(100, 100)形状
signal_efficiency_grid = signal_efficiency.reshape(MQ_grid_14_os.shape)
background_efficiency_grid = background_efficiency.reshape(MQ_grid_14_os.shape)


# 重新计算S_grid和背景
S_grid_14_os_mu = griddata(points_os, values_os, (MQ_grid_14_os, Gstar_grid_14_os), method='linear') * L1*1000 * signal_efficiency_grid * 1000  # 25fb
background_14_os_mu = griddata(points_os, tuple([B_cs]*np.shape(values_os)[0]), (MQ_grid_14_os, Gstar_grid_14_os), method='linear') * L1*1000 * background_efficiency_grid * 1000  # 25fb

# 计算显著度Z = S / sqrt(S + B)
Z_grid_14_os_mu = S_grid_14_os_mu / np.sqrt(S_grid_14_os_mu + background_14_os_mu)

# 创建30fb情况
# 用30fb进行计算
S_grid_14_os_30_mu = griddata(points_os, values_os, (MQ_grid_14_os, Gstar_grid_14_os), method='linear') * L2*1000 * signal_efficiency_grid * 1000  
background_14_os_30_mu = griddata(points_os, tuple([B_cs]*np.shape(values_os)[0]), (MQ_grid_14_os, Gstar_grid_14_os), method='linear') * L2*1000 * background_efficiency_grid * 1000 


# 计算显著度Z = S / sqrt(S + B)
Z_grid_14_os_30_mu = S_grid_14_os_30_mu / np.sqrt(S_grid_14_os_30_mu + background_14_os_30_mu)

def sigma_Z(s, b):
    term1 = (s**2 + 4*b*s + 4*b**2)*s / (4 * (s + b)**3)
    term2 = (s**2 * b) / (4 * (s + b)**3)
    return np.sqrt(term1 + term2)

std_3_mu=sigma_Z(S_grid_14_os_mu,background_14_os_mu)
#Z_3_amu= Z_grid_14_os_amu / np.sqrt(std_3_amu)
z3mup = Z_grid_14_os_mu+np.sqrt(std_3_mu)
z3mum = Z_grid_14_os_mu-np.sqrt(std_3_mu)

std_30_mu=sigma_Z(S_grid_14_os_30_mu,background_14_os_30_mu)
#Z_30_amu= Z_grid_14_os_30_amu / np.sqrt(std_30_amu)
z30mup = Z_grid_14_os_30_mu+np.sqrt(std_30_mu)
z30mum = Z_grid_14_os_30_mu-np.sqrt(std_30_mu)

# 绘制单张图包含两个等值线图
plt.figure(figsize=(8, 6))

# Signal yield
N_prompt_150 = xs  * 150 * 10**(6)  # pb \to ab
N_prompt_25 = xs * 25 * 10**(6) # pb \to ab
# Plot settings
labels = [R'Prompt: 150 ab$^{-1}$', R'Prompt: 25 ab$^{-1}$']
handles = []

plt.figure(figsize=(8,6))
N_prompt_150_grid = griddata(points_os, N_prompt_150.values, (MQ_grid_14_os, Gstar_grid_14_os), method='linear')
N_prompt_25_grid = griddata(points_os, N_prompt_25.values, (MQ_grid_14_os, Gstar_grid_14_os), method='linear')



# Z_grid_14_os_30
contour1 = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, Z_grid_14_os_30_mu, levels=[2], colors='darkorange')
#plt.clabel(contour1, fmt={2: fmt2}, inline=True, fontsize=10)
contour_mu30p = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, z30mup, levels=[2], colors='darkorange',linestyles='--')
contour_mu30m = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, z30mum, levels=[2], colors='darkorange',linestyles='--')
upper_segments_30 = contour_mu30p.allsegs[0]
lower_segments_30 = contour_mu30m.allsegs[0]
for upper_segment, lower_segment in zip(upper_segments_30, lower_segments_30):
    upper_coords = np.array(upper_segment)
    lower_coords = np.array(lower_segment)
    full_coords = np.vstack([upper_coords, lower_coords[::-1]])
    plt.fill(full_coords[:, 0], full_coords[:, 1], color='#ffc680', alpha=0.5)

# Z_grid_14_os
contour2 = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, Z_grid_14_os_mu, levels=[2], colors='navy')
#plt.clabel(contour2, fmt={2: fmt1}, inline=True, fontsize=10,manual=[(70,2e-3)])
contour_mu3p = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, z3mup, levels=[2], colors='navy',linestyles='--')
contour_mu3m = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, z3mum, levels=[2], colors='navy',linestyles='--')
upper_segments_3 = contour_mu3p.allsegs[0]
lower_segments_3 = contour_mu3m.allsegs[0]
for upper_segment, lower_segment in zip(upper_segments_3, lower_segments_3):
    upper_coords = np.array(upper_segment)
    lower_coords = np.array(lower_segment)
    full_coords = np.vstack([upper_coords, lower_coords[::-1]])
    plt.fill(full_coords[:, 0], full_coords[:, 1], color='#8080c0', alpha=0.5)

#plt.title(r'ALP$\rightarrow \tau\tau \rightarrow e \mu $')
plt.ylim(1e-2,1e0)
plt.xlim(1,92)
plt.xscale('log')
plt.xlabel(R"m$_{a}$ [GeV]", fontsize=20)
plt.ylabel(R"$1/\Lambda$ [TeV]$^{-1}$", fontsize=20)
plt.axvline(x=mtau2, linewidth=1.5)
plt.yscale('log')
plt.tick_params(axis='x', labelsize=16)  # x 轴刻度字体大小
plt.tick_params(axis='y', labelsize=16)  # y 轴刻度字体大小
plt.grid()
legend_elements = [
    Line2D([0], [0], color='navy', lw=2, label=R'2$\sigma$ C.L. 25 ab$^{-1}$'),
    Line2D([0], [0], color='darkorange', lw=2, label=R'2$\sigma$ C.L. 150 ab$^{-1}$ '),
    Line2D([0], [0], color='navy', lw=2, linestyle='--', label=R'25 ab$^{-1}$ 1$\sigma$ Error boundary'),
    Line2D([0], [0], color='darkorange', lw=2, linestyle='--', label=R'150 ab$^{-1}$ 1$\sigma$ Error boundary')
]
plt.text(0.6, 0.7, R'$a \to \tau^{+}(\tau^{+}\to \mu^{+} \bar{v}_{\tau} v_{\mu}) \tau^{-}(\tau^{-}\to \mu^{-} v_{\tau} \bar{v}_{\mu})$', ha='center', va='center', transform=plt.gca().transAxes, fontsize=14, color='black')
plt.legend(handles=legend_elements, loc='best', fontsize=10)
plt.savefig('./fignew/taumumu.png',dpi=300)
plt.show()

In [ ]:
from matplotlib.lines import Line2D
z_3_sum=(Z_grid_14_os_e + Z_grid_14_os_emu + Z_grid_14_os_mu)
z_30_sum=(Z_grid_14_os_30_e + Z_grid_14_os_30_emu + Z_grid_14_os_30_mu)

def sigma_Z(s, b):
    term1 = (s**2 + 4*b*s + 4*b**2)*s / (4 * (s + b)**3)
    term2 = (s**2 * b) / (4 * (s + b)**3)
    return np.sqrt(term1 + term2)
std_3_e=sigma_Z(S_grid_14_os_e,background_14_os_e)
std_3_emu=sigma_Z(S_grid_14_os_emu,background_14_os_emu)
std_3_mu=sigma_Z(S_grid_14_os_mu,background_14_os_mu)
std2sum3=std_3_emu**2+std_3_e**2+std_3_mu**2

std_30_e=sigma_Z(S_grid_14_os_30_e,background_14_os_30_e)
std_30_emu=sigma_Z(S_grid_14_os_30_emu,background_14_os_30_emu)
std_30_mu=sigma_Z(S_grid_14_os_30_mu,background_14_os_30_mu)
std2sum30=std_30_emu**2+std_30_e**2+std_30_mu**2

sigma_tot3  = np.sqrt(std2sum3)
sigma_tot30 = np.sqrt(std2sum30)

# ----- 显著度：安全除法 ------------------------------
Z_3_comb  = np.divide(z_3_sum , sigma_tot3 ,
                      out=np.zeros_like(z_3_sum ), where=sigma_tot3  > 0)
Z_30_comb = np.divide(z_30_sum, sigma_tot30,
                      out=np.zeros_like(z_30_sum), where=sigma_tot30 > 0)

# ----- ±1σ 带 ----------------------------------------
z3p  = Z_3_comb  + sigma_tot3
z3m  = Z_3_comb  - sigma_tot3
z30p = Z_30_comb + sigma_tot30
z30m = Z_30_comb - sigma_tot30


plt.figure(figsize=(8, 6))

# Z_grid_14_os_30
contour_comb30 = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, Z_30_comb, levels=[2], colors='darkorange')
contour_comb30p = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, z30p, levels=[2], colors='darkorange',linestyles='--')
contour_comb30m = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, z30m, levels=[2], colors='darkorange',linestyles='--')
upper_segments_30 = contour_comb30p.allsegs[0]
lower_segments_30 = contour_comb30m.allsegs[0]
for upper_segment, lower_segment in zip(upper_segments_30, lower_segments_30):
    upper_coords = np.array(upper_segment)
    lower_coords = np.array(lower_segment)
    full_coords = np.vstack([upper_coords, lower_coords[::-1]])
    plt.fill(full_coords[:, 0], full_coords[:, 1], color='#ffc680', alpha=0.5)



contour_comb3 = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, Z_3_comb, levels=[2], colors='navy')
contour_comb3p = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, z3p, levels=[2], colors='navy',linestyles='--')
contour_comb3m = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, z3m, levels=[2], colors='navy',linestyles='--')
upper_segments_3 = contour_comb3p.allsegs[0]
lower_segments_3 = contour_comb3m.allsegs[0]
for upper_segment, lower_segment in zip(upper_segments_3, lower_segments_3):
    upper_coords = np.array(upper_segment)
    lower_coords = np.array(lower_segment)
    full_coords = np.vstack([upper_coords, lower_coords[::-1]])
    plt.fill(full_coords[:, 0], full_coords[:, 1], color='#8080c0', alpha=0.5)
#plt.clabel(contour_comb3, fmt={2: r'Z=2 (3fb$^{-1}$)'}, inline=True, fontsize=10,manual=[(40,2e-2)])
# plt.clabel(contour_comb3p, fmt={2: r'Zp=2 (3fb$^{-1}$)'}, inline=True, fontsize=10)
# plt.clabel(contour_comb3m, fmt={2: r'Zm=2 (3fb$^{-1}$)'}, inline=True, fontsize=10)
plt.xlim(1,92)
plt.ylim(1e-4,1e0)
plt.xlabel(R"m$_{a}$ [GeV]", fontsize=20)
plt.ylabel(R"$1/\Lambda$ [TeV]$^{-1}$", fontsize=20)
#plt.axvline(x=mtau2, linewidth=1.5, label=R'$2m_{\tau}$')
plt.yscale('log')
plt.xscale('log')
plt.tick_params(axis='x', labelsize=16)  # x 轴刻度字体大小
plt.tick_params(axis='y', labelsize=16)  # y 轴刻度字体大小
plt.grid()
legend_elements = [
    Line2D([0], [0], color='navy', lw=2, label=R'2$\sigma$ C.L. 25 ab$^{-1}$'),
    Line2D([0], [0], color='darkorange', lw=2, label=R'2$\sigma$ C.L. 150 ab$^{-1}$ '),
    Line2D([0], [0], color='navy', lw=2, linestyle='--', label=R'25 ab$^{-1}$ 1$\sigma$ Error boundary'),
    Line2D([0], [0], color='darkorange', lw=2, linestyle='--', label=R'150 ab$^{-1}$ 1$\sigma$ Error boundary')
]
plt.text(0.6, 0.7, R'combined $a \to \tau^{+}\tau^{-}$ channels', ha='center', va='center', transform=plt.gca().transAxes, fontsize=14, color='black')
plt.legend(handles=legend_elements, loc='best', fontsize=10)
plt.savefig('./fignew/combined_tata.png',dpi=300)
plt.show()

In [ ]:
ma = [10, 30, 50, 70, 85]
S_eff = [0.964, 0.912, 0.849, 0.691, 0.314]
B_eff = [0.013, 0.013, 0.013, 0.013, 0.013]
B_cs = 84.29
mmu2=0.1057*2
# 读取数据
pho_ALP = pd.read_csv('./alp_decay_widths_had2.csv')
# 将数据类型转换为更小的浮动数类型以减少内存使用
pho_ALP_mumu = pho_ALP['xs_mumu_prompt'].astype('float32')
pho_ALP_lambda = 1000 / pho_ALP['Lambda']

xs = pho_ALP_mumu
gstar = pho_ALP_lambda
MQ = pho_ALP['ma']

# 为信号截面插值，生成从7到79GeV每隔2GeV的数据点
#signal_xs_interp = np.interp(pho_ALP['ma'], ma, S_eff)
mq_14_os = np.logspace(np.log10(pho_ALP['ma'].min()), np.log10(pho_ALP['ma'].max()), 2000)
gstar_14_os = np.logspace(np.log10(gstar.min()), np.log10(gstar.max()), 200)
# 创建网格，减少网格密度以降低内存使用
#mq_14_os = np.linspace(pho_ALP['ma'].min(), pho_ALP['ma'].max(), 2000)
#gstar_14_os = np.linspace(gstar.min(), gstar.max(), 200)

# 创建网格
MQ_grid_14_os, Gstar_grid_14_os = np.meshgrid(mq_14_os, gstar_14_os)

# 构造插值用的点和目标点
points_frac = np.vstack((MQ, gstar)).T

# 使用二维插值来计算网格上的 S 和 Z
points_os = MQ,gstar.values
values_os = xs.values

# 对每个点进行信号和背景效率的插值
signal_efficiency = np.interp(MQ_grid_14_os.flatten(), ma, S_eff)
background_efficiency = np.interp(MQ_grid_14_os.flatten(), ma, B_eff)

# 将效率的插值结果重塑为(100, 100)形状
signal_efficiency_grid = signal_efficiency.reshape(MQ_grid_14_os.shape)
background_efficiency_grid = background_efficiency.reshape(MQ_grid_14_os.shape)


# 重新计算S_grid和背景
S_grid_14_os_amu = griddata(points_os, values_os, (MQ_grid_14_os, Gstar_grid_14_os), method='linear')  * L1*1000 * signal_efficiency_grid * 1000  # 25fb
background_14_os_amu = griddata(points_os, tuple([B_cs]*np.shape(values_os)[0]), (MQ_grid_14_os, Gstar_grid_14_os), method='linear') * L1*1000 * background_efficiency_grid * 1000  # 25fb

# 计算显著度Z = S / sqrt(S + B)
Z_grid_14_os_amu = S_grid_14_os_amu / np.sqrt(S_grid_14_os_amu + background_14_os_amu)

# 创建30fb情况
# 用30fb进行计算
S_grid_14_os_30_amu = griddata(points_os, values_os, (MQ_grid_14_os, Gstar_grid_14_os), method='linear')  * L2*1000 * signal_efficiency_grid * 1000  
background_14_os_30_amu = griddata(points_os, tuple([B_cs]*np.shape(values_os)[0]), (MQ_grid_14_os, Gstar_grid_14_os), method='linear') * L2*1000 * background_efficiency_grid * 1000 


# 计算显著度Z = S / sqrt(S + B)
Z_grid_14_os_30_amu = S_grid_14_os_30_amu / np.sqrt(S_grid_14_os_30_amu + background_14_os_30_amu)

def sigma_Z(s, b):
    term1 = (s**2 + 4*b*s + 4*b**2)*s / (4 * (s + b)**3)
    term2 = (s**2 * b) / (4 * (s + b)**3)
    return np.sqrt(term1 + term2)

std_3_amu=sigma_Z(S_grid_14_os_amu,background_14_os_amu)
#Z_3_amu= Z_grid_14_os_amu / np.sqrt(std_3_amu)
z3amup = Z_grid_14_os_amu+np.sqrt(std_3_amu)
z3amum = Z_grid_14_os_amu-np.sqrt(std_3_amu)

std_30_amu=sigma_Z(S_grid_14_os_30_amu,background_14_os_30_amu)
#Z_30_amu= Z_grid_14_os_30_amu / np.sqrt(std_30_amu)
z30amup = Z_grid_14_os_30_amu+np.sqrt(std_30_amu)
z30amum = Z_grid_14_os_30_amu-np.sqrt(std_30_amu)

# 绘制单张图包含两个等值线图
plt.figure(figsize=(8, 6))

# Z_grid_14_os_30
contour1 = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, Z_grid_14_os_30_amu, levels=[2], colors='darkorange')
#plt.clabel(contour1, fmt={2: fmt2}, inline=True, fontsize=10)
contour_amu30p = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, z30amup, levels=[2], colors='darkorange',linestyles='--')
contour_amu30m = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, z30amum, levels=[2], colors='darkorange',linestyles='--')
upper_segments_30 = contour_amu30p.allsegs[0]
lower_segments_30 = contour_amu30m.allsegs[0]
for upper_segment, lower_segment in zip(upper_segments_30, lower_segments_30):
    upper_coords = np.array(upper_segment)
    lower_coords = np.array(lower_segment)
    full_coords = np.vstack([upper_coords, lower_coords[::-1]])
    plt.fill(full_coords[:, 0], full_coords[:, 1], color='#ffc680', alpha=0.5)

# Z_grid_14_os
contour2 = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, Z_grid_14_os_amu, levels=[2], colors='navy')
#plt.clabel(contour2, fmt={2: fmt1}, inline=True, fontsize=10,manual=[(70,2e-3)])
contour_amu3p = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, z3amup, levels=[2], colors='navy',linestyles='--')
contour_amu3m = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, z3amum, levels=[2], colors='navy',linestyles='--')
upper_segments_3 = contour_amu3p.allsegs[0]
lower_segments_3 = contour_amu3m.allsegs[0]
for upper_segment, lower_segment in zip(upper_segments_3, lower_segments_3):
    upper_coords = np.array(upper_segment)
    lower_coords = np.array(lower_segment)
    full_coords = np.vstack([upper_coords, lower_coords[::-1]])
    plt.fill(full_coords[:, 0], full_coords[:, 1], color='#8080c0', alpha=0.5)

#plt.title(r'ALP$\rightarrow \tau\tau \rightarrow e \mu $')
plt.ylim(1e-2,1e1)
plt.xlim(0.1,92)
#plt.axvline(x=mmu2, linewidth=1.5)
#plt.axvline(x=mtau2, linewidth=1.5)
plt.xlabel(R"m$_{a}$ [GeV]", fontsize=20)
plt.ylabel(R"$1/\Lambda$ [TeV]$^{-1}$", fontsize=20)
plt.yscale('log')
plt.xscale('log')
plt.tick_params(axis='x', labelsize=16)  # x 轴刻度字体大小
plt.tick_params(axis='y', labelsize=16)  # y 轴刻度字体大小
plt.grid()
legend_elements = [
    Line2D([0], [0], color='navy', lw=2, label=R'2$\sigma$ C.L. 25 ab$^{-1}$'),
    Line2D([0], [0], color='darkorange', lw=2, label=R'2$\sigma$ C.L. 150 ab$^{-1}$ '),
    Line2D([0], [0], color='navy', lw=2, linestyle='--', label=R'25 ab$^{-1}$ 1$\sigma$ Error boundary'),
    Line2D([0], [0], color='darkorange', lw=2, linestyle='--', label=R'150 ab$^{-1}$ 1$\sigma$ Error boundary')
]
plt.text(0.5, 0.7, R'$a \to \mu^{+} \mu^{-}$', ha='center', va='center', transform=plt.gca().transAxes, fontsize=14, color='black')
plt.legend(handles=legend_elements, loc='best', fontsize=10)
plt.savefig('./fignew/amumu.png',dpi=300)
plt.show()


In [ ]:
from matplotlib.lines import Line2D
z_3_sum=(Z_grid_14_os_e + Z_grid_14_os_emu + Z_grid_14_os_mu + Z_grid_14_os_amu)
z_30_sum=(Z_grid_14_os_30_e + Z_grid_14_os_30_emu + Z_grid_14_os_30_mu+Z_grid_14_os_30_amu)

def sigma_Z(s, b):
    term1 = (s**2 + 4*b*s + 4*b**2)*s / (4 * (s + b)**3)
    term2 = (s**2 * b) / (4 * (s + b)**3)
    return np.sqrt(term1 + term2)
std_3_e=sigma_Z(S_grid_14_os_e,background_14_os_e)
std_3_emu=sigma_Z(S_grid_14_os_emu,background_14_os_emu)
std_3_mu=sigma_Z(S_grid_14_os_mu,background_14_os_mu)
std_3_amu=sigma_Z(S_grid_14_os_amu,background_14_os_amu)

std2sum3=std_3_emu**2+std_3_e**2+std_3_mu**2+std_3_amu**2
std_30_e=sigma_Z(S_grid_14_os_30_e,background_14_os_30_e)
std_30_emu=sigma_Z(S_grid_14_os_30_emu,background_14_os_30_emu)
std_30_mu=sigma_Z(S_grid_14_os_30_mu,background_14_os_30_mu)
std_30_amu=sigma_Z(S_grid_14_os_30_amu,background_14_os_30_amu)
std2sum30=std_30_emu**2+std_30_e**2+std_30_mu**2+std_30_amu**2

sigma_tot3  = np.sqrt(std2sum3)
sigma_tot30 = np.sqrt(std2sum30)

# ----- 显著度：安全除法 ------------------------------
Z_3_comb  = np.divide(z_3_sum , sigma_tot3 ,
                      out=np.zeros_like(z_3_sum ), where=sigma_tot3  > 0)
Z_30_comb = np.divide(z_30_sum, sigma_tot30,
                      out=np.zeros_like(z_30_sum), where=sigma_tot30 > 0)

# ----- ±1σ 带 ----------------------------------------
z3p  = Z_3_comb  + sigma_tot3
z3m  = Z_3_comb  - sigma_tot3
z30p = Z_30_comb + sigma_tot30
z30m = Z_30_comb - sigma_tot30


plt.figure(figsize=(8, 6))

# Z_grid_14_os_30
contour_comb30 = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, Z_30_comb, levels=[2], colors='darkorange')
contour_comb30p = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, z30p, levels=[2], colors='darkorange',linestyles='--')
contour_comb30m = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, z30m, levels=[2], colors='darkorange',linestyles='--')
upper_segments_30 = contour_comb30p.allsegs[0]
lower_segments_30 = contour_comb30m.allsegs[0]
for upper_segment, lower_segment in zip(upper_segments_30, lower_segments_30):
    upper_coords = np.array(upper_segment)
    lower_coords = np.array(lower_segment)
    full_coords = np.vstack([upper_coords, lower_coords[::-1]])
    plt.fill(full_coords[:, 0], full_coords[:, 1], color='#ffc680', alpha=0.5)



contour_comb3 = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, Z_3_comb, levels=[2], colors='navy')
contour_comb3p = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, z3p, levels=[2], colors='navy',linestyles='--')
contour_comb3m = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, z3m, levels=[2], colors='navy',linestyles='--')
upper_segments_3 = contour_comb3p.allsegs[0]
lower_segments_3 = contour_comb3m.allsegs[0]
for upper_segment, lower_segment in zip(upper_segments_3, lower_segments_3):
    upper_coords = np.array(upper_segment)
    lower_coords = np.array(lower_segment)
    full_coords = np.vstack([upper_coords, lower_coords[::-1]])
    plt.fill(full_coords[:, 0], full_coords[:, 1], color='#8080c0', alpha=0.5)
#plt.clabel(contour_comb3, fmt={2: r'Z=2 (3fb$^{-1}$)'}, inline=True, fontsize=10,manual=[(40,2e-2)])
# plt.clabel(contour_comb3p, fmt={2: r'Zp=2 (3fb$^{-1}$)'}, inline=True, fontsize=10)
# plt.clabel(contour_comb3m, fmt={2: r'Zm=2 (3fb$^{-1}$)'}, inline=True, fontsize=10)
plt.xlim(0.1,92)
plt.ylim(1e-4,1e1)
plt.xlabel(R"m$_{a}$ [GeV]", fontsize=20)
plt.ylabel(R"$1/\Lambda$ [TeV]$^{-1}$", fontsize=20)
#plt.axvline(x=mmu2, linewidth=1.5)
plt.yscale('log')
plt.xscale('log')
plt.tick_params(axis='x', labelsize=16)  # x 轴刻度字体大小
plt.tick_params(axis='y', labelsize=16)  # y 轴刻度字体大小
plt.grid()
legend_elements = [
    Line2D([0], [0], color='navy', lw=2, label=R'2$\sigma$ C.L. 25 ab$^{-1}$'),
    Line2D([0], [0], color='darkorange', lw=2, label=R'2$\sigma$ C.L. 150 ab$^{-1}$ '),
    Line2D([0], [0], color='navy', lw=2, linestyle='--', label=R'25 ab$^{-1}$ 1$\sigma$ Error boundary'),
    Line2D([0], [0], color='darkorange', lw=2, linestyle='--', label=R'150 ab$^{-1}$ 1$\sigma$ Error boundary')
]
plt.text(0.6, 0.7, R'combined $a \to \mu^{+}\mu^{-}, \tau^{+}\tau^{-}$ channels', ha='center', va='center', transform=plt.gca().transAxes, fontsize=14, color='black')
plt.legend(handles=legend_elements, loc='best', fontsize=10)
#plt.savefig('./fignew/combined.png',dpi=300)
plt.show()

In [ ]:
from matplotlib.lines import Line2D
z_3_sum=(Z_grid_14_os_e + Z_grid_14_os_emu + Z_grid_14_os_mu + Z_grid_14_os_amu)
z_30_sum=(Z_grid_14_os_30_e + Z_grid_14_os_30_emu + Z_grid_14_os_30_mu+Z_grid_14_os_30_amu)

def sigma_Z(s, b):
    term1 = (s**2 + 4*b*s + 4*b**2)*s / (4 * (s + b)**3)
    term2 = (s**2 * b) / (4 * (s + b)**3)
    return np.sqrt(term1 + term2)
std_3_e=sigma_Z(S_grid_14_os_e,background_14_os_e)
std_3_emu=sigma_Z(S_grid_14_os_emu,background_14_os_emu)
std_3_mu=sigma_Z(S_grid_14_os_mu,background_14_os_mu)
std_3_amu=sigma_Z(S_grid_14_os_amu,background_14_os_amu)

std2sum3=std_3_emu**2+std_3_e**2+std_3_mu**2+std_3_amu**2
std_30_e=sigma_Z(S_grid_14_os_30_e,background_14_os_30_e)
std_30_emu=sigma_Z(S_grid_14_os_30_emu,background_14_os_30_emu)
std_30_mu=sigma_Z(S_grid_14_os_30_mu,background_14_os_30_mu)
std_30_amu=sigma_Z(S_grid_14_os_30_amu,background_14_os_30_amu)
std2sum30=std_30_emu**2+std_30_e**2+std_30_mu**2+std_30_amu**2

sigma_tot3  = np.sqrt(std2sum3)
sigma_tot30 = np.sqrt(std2sum30)

# ----- 显著度：安全除法 ------------------------------
Z_3_comb  = np.divide(z_3_sum , sigma_tot3 ,
                      out=np.zeros_like(z_3_sum ), where=sigma_tot3  > 0)
Z_30_comb = np.divide(z_30_sum, sigma_tot30,
                      out=np.zeros_like(z_30_sum), where=sigma_tot30 > 0)

# ----- ±1σ 带 ----------------------------------------
z3p  = Z_3_comb  + sigma_tot3
z3m  = Z_3_comb  - sigma_tot3
z30p = Z_30_comb + sigma_tot30
z30m = Z_30_comb - sigma_tot30

# ==== 绘图 ====
plt.figure(figsize=(8, 6))

# ==== 单通道 25 ab⁻¹ ====
contour_s25 = plt.contour(MQ_grid_14_osme, Gstar_grid_14_osme, Z_grid_14_os_ME, levels=[2], colors='crimson', linewidths=2)
contour_s25p = plt.contour(MQ_grid_14_osme, Gstar_grid_14_osme, z3MEp, levels=[2], colors='crimson', linestyles='--', linewidths=1)
contour_s25m = plt.contour(MQ_grid_14_osme, Gstar_grid_14_osme, z3MEm, levels=[2], colors='crimson', linestyles='--', linewidths=1)
for up, down in zip(contour_s25p.allsegs[0], contour_s25m.allsegs[0]):
    coords = np.vstack([up, down[::-1]])
    plt.fill(coords[:, 0], coords[:, 1], color='crimson', alpha=0.2)

# ==== 单通道 150 ab⁻¹ ====
contour_s150 = plt.contour(MQ_grid_14_osme, Gstar_grid_14_osme, Z_grid_14_os_30_ME, levels=[2], colors='forestgreen', linewidths=2)
contour_s150p = plt.contour(MQ_grid_14_osme, Gstar_grid_14_osme, z30MEp, levels=[2], colors='forestgreen', linestyles='--', linewidths=1)
contour_s150m = plt.contour(MQ_grid_14_osme, Gstar_grid_14_osme, z30MEm, levels=[2], colors='forestgreen', linestyles='--', linewidths=1)
for up, down in zip(contour_s150p.allsegs[0], contour_s150m.allsegs[0]):
    coords = np.vstack([up, down[::-1]])
    plt.fill(coords[:, 0], coords[:, 1], color='forestgreen', alpha=0.2)

# ==== 多通道组合 25 ab⁻¹ ====
contour_c25 = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, Z_3_comb, levels=[2], colors='navy', linewidths=2)
contour_c25p = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, z3p, levels=[2], colors='navy', linestyles='--', linewidths=1)
contour_c25m = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, z3m, levels=[2], colors='navy', linestyles='--', linewidths=1)
for up, down in zip(contour_c25p.allsegs[0], contour_c25m.allsegs[0]):
    coords = np.vstack([up, down[::-1]])
    plt.fill(coords[:, 0], coords[:, 1], color='darkorange', alpha=0.2)

# ==== 多通道组合 150 ab⁻¹ ====
contour_c150 = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, Z_30_comb, levels=[2], colors='darkorange', linewidths=2)
contour_c150p = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, z30p, levels=[2], colors='darkorange', linestyles='--', linewidths=1)
contour_c150m = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, z30m, levels=[2], colors='darkorange', linestyles='--', linewidths=1)
for up, down in zip(contour_c150p.allsegs[0], contour_c150m.allsegs[0]):
    coords = np.vstack([up, down[::-1]])
    plt.fill(coords[:, 0], coords[:, 1], color='navy', alpha=0.2)

# ==== 坐标轴设置 ====
plt.xscale('log')
plt.yscale('log')
plt.xlim(0.1, 90)
plt.ylim(1e-4, 1e0)
#plt.axvline(x=mmu2, linewidth=1.5)
plt.xlabel(R"m$_{a}$ [GeV]", fontsize=20)
plt.ylabel(R"$1/\Lambda$ [TeV]$^{-1}$", fontsize=20)
plt.tick_params(axis='both', labelsize=16)
plt.grid()

# ==== 图例 ====
legend_elements = [
    Line2D([0], [0], color='navy', lw=2, label='Prompt Lepton Signal 25 ab$^{-1}$'),
    Line2D([0], [0], color='darkorange', lw=2, label='Prompt Lepton Signal 150 ab$^{-1}$'),
    Line2D([0], [0], color='crimson', lw=2, label=R'Mono-$\gamma$ Signal 25 ab$^{-1}$'),
    Line2D([0], [0], color='forestgreen', lw=2, label=R'Mono-$\gamma$ Signal 150 ab$^{-1}$')
]
plt.legend(handles=legend_elements, fontsize=10)

#plt.text(0.6, 0.5, R'Combined $a \to \mu\mu,\ \tau\tau$ channels', transform=plt.gca().transAxes, ha='center', va='center', fontsize=14, color='black')

plt.savefig('./fignew/combined_all_contours.png', dpi=300)
plt.show()

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Polygon

# ============================================================
# 0) 读取 limit CSV：保持原始顺序 + 自动分段（用于 log 轴）
# ============================================================

def load_polyline_segments(csv_path):
    """
    读取两列无表头 CSV: x=m_a, y=1/Lambda
    返回若干段 polyline (x_seg, y_seg)，保持文件原始顺序。
    分段规则：遇到 NaN / 非有限 / 非正值 就断开。
    """
    arr = pd.read_csv(csv_path, header=None).values
    x = np.asarray(arr[:, 0], dtype=float)
    y = np.asarray(arr[:, 1], dtype=float)

    good = np.isfinite(x) & np.isfinite(y) & (x > 0) & (y > 0)

    segs = []
    start = None
    for i, ok in enumerate(good):
        if ok and start is None:
            start = i
        if (not ok) and (start is not None):
            xs = x[start:i]
            ys = y[start:i]
            if len(xs) >= 2:
                segs.append((xs, ys))
            start = None
    if start is not None:
        xs = x[start:]
        ys = y[start:]
        if len(xs) >= 2:
            segs.append((xs, ys))

    # 去掉每段里相邻重复点（可选，但对避免奇怪尖刺有帮助）
    cleaned = []
    for xs, ys in segs:
        dx = np.diff(xs)
        dy = np.diff(ys)
        keep = np.ones(len(xs), dtype=bool)
        keep[1:] = ~((dx == 0) & (dy == 0))
        xs2, ys2 = xs[keep], ys[keep]
        if len(xs2) >= 2:
            cleaned.append((xs2, ys2))

    return cleaned


# ============================================================
# 1) 叠加 limits：画线 + 填充（B/Charm/KpiX/SN1987A）
# ============================================================

def overlay_limits(ax, limit_dir="./limit",
                   x_left=0.1, x_right=90.0,
                   y_bottom=1e-4, y_top=1.0,
                   fill_alpha=0.20,
                   line_lw=1.6,
                   z_fill=0.6, z_line=1.2,
                   line_color_map=None):
    files = {
        "B": "B.csv",
        "Charm": "charm.csv",
        "KpiX": "KpiX.csv",
        "leP": "Lep.csv",
        "LHC": "LHC.csv",
        "SN1987A": "SN1987A.csv",
    }

    if line_color_map is None:
        line_color_map = {}

    # --- 先画线（分段画；legend 用第一段的句柄） ---
    line_objs = {}
    line_colors = {}

    for lab, fn in files.items():
        path = os.path.join(limit_dir, fn)
        if not os.path.exists(path):
            raise FileNotFoundError(f"找不到：{path}")

        segs = load_polyline_segments(path)

        desired_color = line_color_map.get(lab, None)

        first_handle = None
        for xs, ys in segs:
            # 这里先裁剪到视窗范围：用 NaN 让 Matplotlib 自动断线
            m = (xs >= x_left) & (xs <= x_right) & (ys >= y_bottom) & (ys <= y_top)
            xs_plot = xs.copy()
            ys_plot = ys.copy()
            xs_plot[~m] = np.nan
            ys_plot[~m] = np.nan

            kwargs = dict(lw=line_lw, zorder=z_line)
            if desired_color is not None:
                kwargs["color"] = desired_color

            h, = ax.loglog(xs_plot, ys_plot, **kwargs)

            if first_handle is None:
                first_handle = h

        if first_handle is None:
            kwargs = dict(lw=line_lw, zorder=z_line)
            if desired_color is not None:
                kwargs["color"] = desired_color
            first_handle, = ax.loglog([], [], **kwargs)

        first_handle.set_label(lab)
        line_objs[lab] = first_handle

        # 关键：如果指定了颜色，就用指定的；否则沿用 matplotlib 自动颜色
        line_colors[lab] = desired_color if desired_color is not None else first_handle.get_color()

    # --- 填充：优先用 shapely 做严格裁剪 + 分段 ---
    filled_patches = []

    try:
        from shapely.geometry import LineString
        from shapely.ops import clip_by_rect

        def clip_segments(xs, ys):
            line = LineString(np.column_stack([xs, ys]))
            clipped = clip_by_rect(line, x_left, y_bottom, x_right, y_top)
            segs_out = []
            if clipped.is_empty:
                return segs_out
            if clipped.geom_type == "LineString":
                c = np.array(clipped.coords)
                segs_out.append((c[:, 0], c[:, 1]))
            elif clipped.geom_type == "MultiLineString":
                for g in clipped.geoms:
                    c = np.array(g.coords)
                    segs_out.append((c[:, 0], c[:, 1]))
            return segs_out

        def touches_left(xs, tol=1e-12):
            return np.any(np.isclose(xs, x_left, atol=tol, rtol=0))

        def touches_top(ys, tol=1e-12):
            return np.any(np.isclose(ys, y_top, atol=tol, rtol=0))

        def add_fill(xs, ys, mode, color, lw_outline=1.6):
            xs = np.asarray(xs); ys = np.asarray(ys)
            if len(xs) < 2:
                return None, None

            if mode == "left":
                verts = [(x_left, ys[0])] + list(zip(xs, ys)) + [(x_left, ys[-1])]
            elif mode == "top":
                verts = [(xs[0], y_top)] + list(zip(xs, ys)) + [(xs[-1], y_top)]
            elif mode == "upper_left":
                verts = [(x_left, y_top), (x_left, ys[0])] + list(zip(xs, ys)) + [(xs[-1], y_top)]
            else:
                return None, None

            # 填充面
            poly = Polygon(
                verts, closed=True,
                facecolor=color, edgecolor="none",
                alpha=fill_alpha, zorder=z_fill
            )
            ax.add_patch(poly)

            # 闭合轮廓线（把闭合边也画出来）
            vx = [p[0] for p in verts] + [verts[0][0]]
            vy = [p[1] for p in verts] + [verts[0][1]]
            (outline,) = ax.loglog(
                vx, vy, color=color, lw=lw_outline,
                zorder=z_line, solid_capstyle="round"
            )

            return poly, outline

        for lab in ["B", "Charm", "KpiX", "SN1987A"]:
            path = os.path.join(limit_dir, files[lab])
            segs = load_polyline_segments(path)

            for xs, ys in segs:
                for xc, yc in clip_segments(xs, ys):
                    if lab == "SN1987A":
                        p = add_fill(xc, yc, "left", line_colors[lab])
                    elif lab == "B":
                        p = add_fill(xc, yc, "top", line_colors[lab])
                    elif lab == "Charm":
                        p = add_fill(xc, yc, "upper_left", line_colors[lab])
                    elif lab == "KpiX":
                        L = touches_left(xc)
                        T = touches_top(yc)
                        if L and T:
                            p = add_fill(xc, yc, "upper_left", line_colors[lab])
                        elif L:
                            p = add_fill(xc, yc, "left", line_colors[lab])
                        elif T:
                            p = add_fill(xc, yc, "top", line_colors[lab])
                        else:
                            p = None
                    else:
                        p = None

                    if p is not None:
                        filled_patches.append((lab, p))

    except Exception:
        # shapely 不可用：近似填充（不做严格裁剪）
        def add_fill_approx(xs, ys, mode, color):
            m = (xs >= x_left) & (xs <= x_right) & (ys >= y_bottom) & (ys <= y_top)
            xs2 = xs[m]; ys2 = ys[m]
            if len(xs2) < 2:
                return None

            if mode == "left":
                verts = [(x_left, ys2[0])] + list(zip(xs2, ys2)) + [(x_left, ys2[-1])]
            elif mode == "top":
                verts = [(xs2[0], y_top)] + list(zip(xs2, ys2)) + [(xs2[-1], y_top)]
            elif mode == "upper_left":
                verts = [(x_left, y_top), (x_left, ys2[0])] + list(zip(xs2, ys2)) + [(xs2[-1], y_top)]
            else:
                return None

            poly = Polygon(
                verts, closed=True,
                facecolor=color, edgecolor="none",
                alpha=fill_alpha, zorder=z_fill
            )
            ax.add_patch(poly)
            return poly

        for lab in ["B", "Charm", "KpiX", "SN1987A"]:
            path = os.path.join(limit_dir, files[lab])
            segs = load_polyline_segments(path)

            for xs, ys in segs:
                if lab == "SN1987A":
                    p = add_fill_approx(xs, ys, "left", line_colors[lab])
                elif lab == "B":
                    p = add_fill_approx(xs, ys, "top", line_colors[lab])
                elif lab == "Charm":
                    p = add_fill_approx(xs, ys, "upper_left", line_colors[lab])
                elif lab == "KpiX":
                    p = add_fill_approx(xs, ys, "upper_left", line_colors[lab])
                else:
                    p = None
                if p is not None:
                    filled_patches.append((lab, p))

    return line_objs, filled_patches



# ============================================================
# 2) 你原先的显著度计算（保持你的写法不改）
# ============================================================

z_3_sum=(Z_grid_14_os_e + Z_grid_14_os_emu + Z_grid_14_os_mu + Z_grid_14_os_amu)
z_30_sum=(Z_grid_14_os_30_e + Z_grid_14_os_30_emu + Z_grid_14_os_30_mu+Z_grid_14_os_30_amu)

def sigma_Z(s, b):
    term1 = (s**2 + 4*b*s + 4*b**2)*s / (4 * (s + b)**3)
    term2 = (s**2 * b) / (4 * (s + b)**3)
    return np.sqrt(term1 + term2)

std_3_e=sigma_Z(S_grid_14_os_e,background_14_os_e)
std_3_emu=sigma_Z(S_grid_14_os_emu,background_14_os_emu)
std_3_mu=sigma_Z(S_grid_14_os_mu,background_14_os_mu)
std_3_amu=sigma_Z(S_grid_14_os_amu,background_14_os_amu)
std2sum3=std_3_emu**2+std_3_e**2+std_3_mu**2+std_3_amu**2

std_30_e=sigma_Z(S_grid_14_os_30_e,background_14_os_30_e)
std_30_emu=sigma_Z(S_grid_14_os_30_emu,background_14_os_30_emu)
std_30_mu=sigma_Z(S_grid_14_os_30_mu,background_14_os_30_mu)
std_30_amu=sigma_Z(S_grid_14_os_30_amu,background_14_os_30_amu)
std2sum30=std_30_emu**2+std_30_e**2+std_30_mu**2+std_30_amu**2

sigma_tot3  = np.sqrt(std2sum3)
sigma_tot30 = np.sqrt(std2sum30)

Z_3_comb  = np.divide(z_3_sum , sigma_tot3 ,
                      out=np.zeros_like(z_3_sum ), where=sigma_tot3  > 0)
Z_30_comb = np.divide(z_30_sum, sigma_tot30,
                      out=np.zeros_like(z_30_sum), where=sigma_tot30 > 0)

z3p  = Z_3_comb  + sigma_tot3
z3m  = Z_3_comb  - sigma_tot3
z30p = Z_30_comb + sigma_tot30
z30m = Z_30_comb - sigma_tot30


# ============================================================
# 3) 作图：先 limits（底层），再你的 contours（上层）
# ============================================================

plt.figure(figsize=(8, 6))
ax = plt.gca()

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlim(0.1, 90)
ax.set_ylim(1e-4, 1e0)

# --- 叠加 limits（关键：不排序 + 分段画，避免尖刺） ---
line_color_map = {
    "B": "#0D47A1",
    "Charm": "#d62728",
    "KpiX": "#6D4C41",
    "SN1987A": "#424242",
}
line_objs, filled = overlay_limits(ax, limit_dir="./limit", line_color_map=line_color_map)


# --- 你的四组 contour（原样） ---
contour_s25 = plt.contour(MQ_grid_14_osme, Gstar_grid_14_osme, Z_grid_14_os_ME, levels=[2], colors='crimson', linewidths=2)
contour_s25p = plt.contour(MQ_grid_14_osme, Gstar_grid_14_osme, z3MEp, levels=[2], colors='crimson', linestyles='--', linewidths=1)
contour_s25m = plt.contour(MQ_grid_14_osme, Gstar_grid_14_osme, z3MEm, levels=[2], colors='crimson', linestyles='--', linewidths=1)
for up, down in zip(contour_s25p.allsegs[0], contour_s25m.allsegs[0]):
    coords = np.vstack([up, down[::-1]])
    plt.fill(coords[:, 0], coords[:, 1], color='crimson', alpha=0.2)

contour_s150 = plt.contour(MQ_grid_14_osme, Gstar_grid_14_osme, Z_grid_14_os_30_ME, levels=[2], colors='forestgreen', linewidths=2)
contour_s150p = plt.contour(MQ_grid_14_osme, Gstar_grid_14_osme, z30MEp, levels=[2], colors='forestgreen', linestyles='--', linewidths=1)
contour_s150m = plt.contour(MQ_grid_14_osme, Gstar_grid_14_osme, z30MEm, levels=[2], colors='forestgreen', linestyles='--', linewidths=1)
for up, down in zip(contour_s150p.allsegs[0], contour_s150m.allsegs[0]):
    coords = np.vstack([up, down[::-1]])
    plt.fill(coords[:, 0], coords[:, 1], color='forestgreen', alpha=0.2)

contour_c25 = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, Z_3_comb, levels=[2], colors='navy', linewidths=2)
contour_c25p = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, z3p, levels=[2], colors='navy', linestyles='--', linewidths=1)
contour_c25m = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, z3m, levels=[2], colors='navy', linestyles='--', linewidths=1)
for up, down in zip(contour_c25p.allsegs[0], contour_c25m.allsegs[0]):
    coords = np.vstack([up, down[::-1]])
    plt.fill(coords[:, 0], coords[:, 1], color='darkorange', alpha=0.2)

contour_c150 = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, Z_30_comb, levels=[2], colors='darkorange', linewidths=2)
contour_c150p = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, z30p, levels=[2], colors='darkorange', linestyles='--', linewidths=1)
contour_c150m = plt.contour(MQ_grid_14_os, Gstar_grid_14_os, z30m, levels=[2], colors='darkorange', linestyles='--', linewidths=1)
for up, down in zip(contour_c150p.allsegs[0], contour_c150m.allsegs[0]):
    coords = np.vstack([up, down[::-1]])
    plt.fill(coords[:, 0], coords[:, 1], color='navy', alpha=0.2)

plt.xlabel(r"m$_{a}$ [GeV]", fontsize=20)
plt.ylabel(r"$1/\Lambda$ [TeV]$^{-1}$", fontsize=20)
plt.tick_params(axis='both', labelsize=16)
#plt.grid(True, which="both", alpha=0.3)

# --- 图例：双图例（信号 + limits） ---
legend_elements = [
    Line2D([0], [0], color='navy', lw=2, label='Prompt Lepton Signal 25 ab$^{-1}$'),
    Line2D([0], [0], color='darkorange', lw=2, label='Prompt Lepton Signal 150 ab$^{-1}$'),
    Line2D([0], [0], color='crimson', lw=2, label=r'Mono-$\gamma$ Signal 25 ab$^{-1}$'),
    Line2D([0], [0], color='forestgreen', lw=2, label=r'Mono-$\gamma$ Signal 150 ab$^{-1}$')
]
leg1 = ax.legend(
    handles=legend_elements,
    fontsize=10,
    loc="upper right",
    #bbox_to_anchor=(0.52, 0.55),  # (x,y) 在 [0,1]x[0,1]
    frameon=True
)

ax.add_artist(leg1)

limit_handles = [line_objs[k] for k in ["B", "Charm", "KpiX", "SN1987A"]]
ax.legend(handles=limit_handles, fontsize=9, loc="best",
          title="Existing limits", frameon=True, bbox_to_anchor=(0.62, 0.35))


os.makedirs("./fignew", exist_ok=True)
plt.savefig("./fignew/combined_all_contours_with_limits.png", dpi=300, bbox_inches="tight")
plt.show()
